# Project: quantum versus classical kernels, honestly
**Quantum Machine Learning · Week 2 (Quantum kernels) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qml/w2d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Project: quantum versus classical kernels, honestly.** Same data, same splits, same tuning budget,
one look at each test set, and error bars. Four datasets, four kernels: linear, RBF, and two quantum ones.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.datasets import make_moons, make_circles, load_iris, load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC

def angle_map(x):
    qc = QuantumCircuit(len(x))
    for q, value in enumerate(x):
        qc.ry(value, q)
    return qc

def zz_map(x, reps=2):
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for q in range(n):
            qc.p(2 * x[q], q)
        for q in range(n - 1):
            qc.cx(q, q + 1)
            qc.p(2 * (np.pi - x[q]) * (np.pi - x[q + 1]), q + 1)
            qc.cx(q, q + 1)
    return qc

def datasets():
    X1, y1 = make_moons(n_samples=200, noise=0.2, random_state=0)
    X2, y2 = make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=0)
    iris = load_iris()
    keep = iris.target > 0                                   # versicolor versus virginica: the hard pair
    digits = load_digits()
    mask = np.isin(digits.target, [3, 8])
    return {"moons": (X1, y1), "circles": (X2, y2),
            "iris": (iris.data[keep], (iris.target[keep] == 2).astype(int)),
            "digits 3 vs 8": (digits.data[mask], (digits.target[mask] == 8).astype(int))}

print({name: X.shape for name, (X, y) in datasets().items()})

## The protocol
For every split: shrink to at most 4 features with PCA and scale to [0, 1], both fitted on the training part;
tune every model's knobs by 5-fold cross-validation on the training part only; then test once.

In [ ]:
C_GRID = [0.1, 1, 10, 100]

def cv_best(K, y):
    """Best C and its CV accuracy for a precomputed kernel matrix."""
    best = (-1, None)
    folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(K, y))
    for C in C_GRID:
        score = np.mean([SVC(kernel="precomputed", C=C).fit(K[np.ix_(tr, tr)], y[tr]).score(K[np.ix_(va, tr)], y[va]) for tr, va in folds])
        best = max(best, (score, C))
    return best

def quantum_kernels(fmap, U_tr, U_te, s):
    S_tr = np.array([Statevector(fmap(s * u)).data for u in U_tr])
    S_te = np.array([Statevector(fmap(s * u)).data for u in U_te])
    return np.abs(S_tr.conj() @ S_tr.T) ** 2, np.abs(S_te.conj() @ S_tr.T) ** 2

def classical_kernels(kind, U_tr, U_te, gamma):
    if kind == "linear":
        return U_tr @ U_tr.T, U_te @ U_tr.T
    sq = lambda A, B: ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
    return np.exp(-gamma * sq(U_tr, U_tr)), np.exp(-gamma * sq(U_te, U_tr))

KNOBS = {"linear": [None], "RBF": [0.1, 1, 10, 100], "angle (quantum)": [0.25, 0.5, 1, 2, np.pi], "ZZ (quantum)": [0.25, 0.5, 1, 2, np.pi]}

def run_split(X, y, seed):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    if X.shape[1] > 4:
        pca = PCA(n_components=4).fit(X_tr)
        X_tr, X_te = pca.transform(X_tr), pca.transform(X_te)
    scaler = MinMaxScaler((0, 1)).fit(X_tr)
    U_tr, U_te = scaler.transform(X_tr), scaler.transform(X_te)
    scores = {}
    for model, knobs in KNOBS.items():
        best = (-1, None, None)
        for knob in knobs:
            if model == "linear" or model == "RBF":
                K_tr, K_te = classical_kernels("linear" if model == "linear" else "rbf", U_tr, U_te, knob)
            else:
                K_tr, K_te = quantum_kernels(angle_map if model.startswith("angle") else zz_map, U_tr, U_te, knob)
            cv, C = cv_best(K_tr, y_tr)
            if cv > best[0]:
                best = (cv, C, (K_tr, K_te))
        cv, C, (K_tr, K_te) = best
        scores[model] = SVC(kernel="precomputed", C=C).fit(K_tr, y_tr).score(K_te, y_te)   # the one test
    return scores

## Run the benchmark: 4 datasets, 5 random splits each

In [ ]:
def benchmark(data, seeds=range(5)):
    table = {}
    for name, (X, y) in data.items():
        runs = [run_split(X, y, seed) for seed in seeds]
        table[name] = {model: (np.mean([r[model] for r in runs]), np.std([r[model] for r in runs])) for model in KNOBS}
    return table

table = benchmark(datasets())
print(f"{'dataset':15s}" + "".join(f"{m:>18s}" for m in KNOBS))
for name, row in table.items():
    print(f"{name:15s}" + "".join(f"{row[m][0]:>12.3f} ± {row[m][1]:.3f}" for m in KNOBS))

## Project
1. Read the table. Where do the quantum kernels tie with RBF? Lose? Win by more than the spread?
2. Add a fifth dataset: breast cancer (`load_breast_cancer`, 30 measurements per tumor, shrunk to 4 by the protocol).
3. Find the biggest quantum lead in your table and retest it on 20 fresh splits (seeds 100 to 119):
   the average gap, its spread, and how often the quantum kernel wins, ties and loses.
4. Write three honest sentences: what the evidence says, and what it doesn't.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
extra = {}       # TODO: benchmark({"breast cancer": (cancer.data, cancer.target)}), then print its row

# TODO: retest the biggest lead on fresh splits, for example
# runs = [run_split(X, y, seed) for seed in range(100, 120)]
# gaps = np.array([r["ZZ (quantum)"] - r["RBF"] for r in runs])

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
extra = benchmark({"breast cancer": (cancer.data, cancer.target)})
for name, row in extra.items():
    print(f"{name:15s}" + "".join(f"{row[m][0]:>12.3f} ± {row[m][1]:.3f}" for m in KNOBS))

# the biggest lead: ZZ on breast cancer. Retest it on 20 splits it has never seen.
runs = [run_split(cancer.data, cancer.target, seed) for seed in range(100, 120)]
gaps = np.array([r["ZZ (quantum)"] - r["RBF"] for r in runs])
print(f"\nZZ minus RBF on 20 fresh splits: {100 * gaps.mean():+.1f} ± {100 * gaps.std():.1f} points")
print(f"ZZ wins {np.sum(gaps > 1e-9)}, ties {np.sum(np.abs(gaps) < 1e-9)}, loses {np.sum(gaps < -1e-9)}")
print()
print("1. On five small datasets, both quantum kernels match a tuned RBF kernel within the spread,")
print("   except the angle kernel on moons, which loses by about 12 points.")
print("2. The one lead, ZZ on breast cancer, halved on fresh splits and its spread now covers zero:")
print("   the first estimate was partly luck, picked because it looked best.")
print("3. This is no evidence of a quantum advantage on classical data like these; it says nothing")
print("   about data with quantum structure, which week 4 visits.")